In [ ]:
conda install -c conda-forge folium

Retrieving notices: ...working... done
Channels:
 - conda-forge
 - defaults
Platform: osx-64

In [2]:
import os
print(os.getcwd())

/Users/panping/Friends-map


In [1]:
%%writefile app.py

UsageError: %%writefile is a cell magic, but the cell body is empty.


In [3]:
%%writefile app.py

import streamlit as st

st.title("🌍 我的朋友地圖")
st.write("Streamlit 已成功啟動！")

Writing app.py


In [5]:
%%writefile app.py
from supabase import Client, create_client
supabase = get_supabase_client()
supabase.table("friends").insert(...)
with st.expander("Supabase 連線診斷"):
    project_url = str(st.secrets["SUPABASE_URL"])

    st.write(
        "連線專案：",
        project_url.replace("https://", "").split(".")[0],
    )

    try:
        test_response = (
            supabase
            .table("friends")
            .select("*")
            .limit(5)
            .execute()
        )

        st.success("Supabase 讀取連線成功")
        st.write("目前讀到的資料：", test_response.data)

    except Exception as error:
        st.error("Supabase 讀取連線失敗")
        st.exception(error)

@st.cache_resource
def get_supabase_client() -> Client:
    """建立並快取 Supabase 連線。"""
    return create_client(
        st.secrets["SUPABASE_URL"],
        st.secrets["SUPABASE_KEY"],
    )



import time

import folium
import streamlit as st
from folium.plugins import MarkerCluster
from geopy.exc import GeocoderServiceError, GeocoderTimedOut
from geopy.geocoders import Nominatim
from streamlit_folium import st_folium




# --------------------------------------------------
# 資料庫
# --------------------------------------------------

def add_friend(
    name: str,
    city: str,
    country: str,
    latitude: float,
    longitude: float,
    notes: str,
):
    """新增朋友並回傳 Supabase 結果。"""

    response = (
        supabase
        .table("friends")
        .insert(
            {
                "name": name,
                "city": city,
                "country": country,
                "latitude": latitude,
                "longitude": longitude,
                "notes": notes,
            }
        )
        .execute()
    )

    return response.data

def get_friends(search_text: str = ""):
    """讀取所有朋友或依關鍵字搜尋。"""
    response = (
        supabase
        .table("friends")
        .select("*")
        .order("name")
        .execute()
    )

    friends = response.data or []
    clean_search = search_text.strip().lower()

    if not clean_search:
        return friends

    return [
        friend
        for friend in friends
        if (
            clean_search in str(friend.get("name", "")).lower()
            or clean_search in str(friend.get("city", "")).lower()
            or clean_search in str(friend.get("country", "")).lower()
            or clean_search in str(friend.get("notes", "")).lower()
        )
    ]

def get_friend(friend_id: int):
    """依 ID 取得一位朋友。"""
    response = (
        supabase
        .table("friends")
        .select("*")
        .eq("id", friend_id)
        .limit(1)
        .execute()
    )

    if not response.data:
        return None

    return response.data[0]

def update_friend(
    friend_id: int,
    name: str,
    city: str,
    country: str,
    latitude: float,
    longitude: float,
    notes: str,
) -> None:
    """更新 Supabase 中的朋友資料。"""
    (
        supabase
        .table("friends")
        .update(
            {
                "name": name,
                "city": city,
                "country": country,
                "latitude": latitude,
                "longitude": longitude,
                "notes": notes,
            }
        )
        .eq("id", friend_id)
        .execute()
    )


def delete_friend(friend_id: int) -> None:
    """從 Supabase 刪除朋友。"""
    (
        supabase
        .table("friends")
        .delete()
        .eq("id", friend_id)
        .execute()
    )

# --------------------------------------------------
# 城市定位
# --------------------------------------------------

@st.cache_data(ttl=86400, show_spinner=False)
def geocode_city(city: str, country: str):
    """將城市與國家轉換成經緯度。"""
    geolocator = Nominatim(
        user_agent="personal-friends-map-app"
    )

    search_text = f"{city}, {country}"

    try:
        location = geolocator.geocode(
            search_text,
            exactly_one=True,
            timeout=10,
        )
    except (GeocoderTimedOut, GeocoderServiceError):
        return None

    if location is None:
        return None

    return {
        "latitude": float(location.latitude),
        "longitude": float(location.longitude),
        "display_name": str(location.address),
    }


# --------------------------------------------------
# 地圖
# --------------------------------------------------

def create_map(friends):
    """建立朋友地圖。"""
    if friends:
        average_latitude = sum(
            float(friend["latitude"])
            for friend in friends
        ) / len(friends)

        average_longitude = sum(
            float(friend["longitude"])
            for friend in friends
        ) / len(friends)

        map_location = [
            average_latitude,
            average_longitude,
        ]

        zoom_start = 2 if len(friends) > 1 else 7
    else:
        map_location = [20, 0]
        zoom_start = 2

    friends_map = folium.Map(
        location=map_location,
        zoom_start=zoom_start,
        tiles="OpenStreetMap",
    )

    marker_cluster = MarkerCluster().add_to(
        friends_map
    )

    for friend in friends:
        notes = friend.get("notes") or ""

        popup_content = f"""
        <div style="min-width: 180px;">
            <strong>{friend["name"]}</strong><br>
            {friend["city"]}, {friend["country"]}
        """

        if notes:
            popup_content += f"<br><br>{notes}"

        popup_content += "</div>"

        folium.Marker(
            location=[
                float(friend["latitude"]),
                float(friend["longitude"]),
            ],
            tooltip=str(friend["name"]),
            popup=folium.Popup(
                popup_content,
                max_width=300,
            ),
        ).add_to(marker_cluster)

    return friends_map


# --------------------------------------------------
# 頁面設定
# --------------------------------------------------

st.set_page_config(
    page_title="我的朋友地圖",
    page_icon="🌍",
    layout="wide",
)


if "editing_friend_id" not in st.session_state:
    st.session_state.editing_friend_id = None

if "deleting_friend_id" not in st.session_state:
    st.session_state.deleting_friend_id = None


st.title("🌍 我的朋友地圖")
st.caption("記錄世界各地朋友居住的城市。")


# --------------------------------------------------
# 新增朋友
# --------------------------------------------------

with st.expander("➕ 加入朋友", expanded=False):
    with st.form(
        "add_friend_form",
        clear_on_submit=True,
    ):
        name = st.text_input(
            "朋友姓名",
            placeholder="例如：Marie",
        )

        col1, col2 = st.columns(2)

        with col1:
            city = st.text_input(
                "城市",
                placeholder="例如：Paris",
            )

        with col2:
            country = st.text_input(
                "國家或地區",
                placeholder="例如：France",
            )

        notes = st.text_area(
            "備註",
            placeholder="例如：大學同學、WhatsApp 聯絡",
        )

        submitted = st.form_submit_button(
            "搜尋城市並加入",
            use_container_width=True,
        )

        if submitted:
            clean_name = name.strip()
            clean_city = city.strip()
            clean_country = country.strip()
            clean_notes = notes.strip()

            if (
                not clean_name
                or not clean_city
                or not clean_country
            ):
                st.error(
                    "請填寫朋友姓名、城市和國家。"
                )
            else:
                with st.spinner(
                    "正在搜尋城市位置……"
                ):
                    location = geocode_city(
                        clean_city,
                        clean_country,
                    )

                if location is None:
                    st.error(
                        "找不到這個城市。"
                        "請檢查城市和國家名稱。"
                    )
                try:
                    inserted_data = add_friend(
                        name=clean_name,
                        city=clean_city,
                        country=clean_country,
                        latitude=location["latitude"],
                        longitude=location["longitude"],
                        notes=clean_notes,
                    )
                    if inserted_data:
                        st.success(
                            f"已成功寫入 Supabase: {clean_name}"
                        )
                        st.write("Supabase 回傳：", inserted_data)
                    else:
                        st.error(
                            "Supabase 沒有回傳新增資料，請檢查RTS、API Key 和專案網址。"
                        )

                except Exception as error:
                    st.error("Supabase 寫入失敗")
                    st.exception(error)


# --------------------------------------------------
# 搜尋
# --------------------------------------------------

st.subheader("🔍 搜尋朋友")

search_text = st.text_input(
    "輸入姓名、城市、國家或備註",
    placeholder="例如：Marie、Paris、Japan",
    label_visibility="collapsed",
)

friends = get_friends(search_text)

if search_text:
    st.caption(
        f"找到 {len(friends)} 筆符合「{search_text}」的資料"
    )
else:
    st.caption(
        f"目前共有 {len(friends)} 位朋友"
    )


# --------------------------------------------------
# 地圖
# --------------------------------------------------

st.subheader("🗺️ 朋友地圖")

friends_map = create_map(friends)

st_folium(
    friends_map,
    height=550,
    use_container_width=True,
    key="friends-map",
)


# --------------------------------------------------
# 編輯表單
# --------------------------------------------------

if st.session_state.editing_friend_id is not None:
    editing_friend = get_friend(
        st.session_state.editing_friend_id
    )

    if editing_friend:
        st.divider()
        st.subheader(
            f"✏️ 編輯 {editing_friend['name']}"
        )

        with st.form("edit_friend_form"):
            edited_name = st.text_input(
                "朋友姓名",
                value=editing_friend["name"],
            )

            col1, col2 = st.columns(2)

            with col1:
                edited_city = st.text_input(
                    "城市",
                    value=editing_friend["city"],
                )

            with col2:
                edited_country = st.text_input(
                    "國家或地區",
                    value=editing_friend["country"],
                )

            edited_notes = st.text_area(
                "備註",
                value=editing_friend["notes"] or "",
            )

            save_col, cancel_col = st.columns(2)

            with save_col:
                save_edit = st.form_submit_button(
                    "儲存修改",
                    use_container_width=True,
                    type="primary",
                )

            with cancel_col:
                cancel_edit = st.form_submit_button(
                    "取消",
                    use_container_width=True,
                )

            if cancel_edit:
                st.session_state.editing_friend_id = None
                st.rerun()

            if save_edit:
                clean_name = edited_name.strip()
                clean_city = edited_city.strip()
                clean_country = edited_country.strip()
                clean_notes = edited_notes.strip()

                if (
                    not clean_name
                    or not clean_city
                    or not clean_country
                ):
                    st.error(
                        "姓名、城市和國家不能留空。"
                    )
                else:
                    city_changed = (
                        clean_city.lower()
                        != editing_friend["city"].lower()
                        or clean_country.lower()
                        != editing_friend["country"].lower()
                    )

                    if city_changed:
                        with st.spinner(
                            "正在更新城市位置……"
                        ):
                            location = geocode_city(
                                clean_city,
                                clean_country,
                            )

                        if location is None:
                            st.error(
                                "找不到新的城市位置，"
                                "請檢查城市和國家名稱。"
                            )
                            st.stop()

                        latitude = location["latitude"]
                        longitude = location["longitude"]
                    else:
                        latitude = float(
                            editing_friend["latitude"]
                        )
                        longitude = float(
                            editing_friend["longitude"]
                        )

                    update_friend(
                        friend_id=editing_friend["id"],
                        name=clean_name,
                        city=clean_city,
                        country=clean_country,
                        latitude=latitude,
                        longitude=longitude,
                        notes=clean_notes,
                    )

                    st.session_state.editing_friend_id = None
                    st.success("朋友資料已更新。")
                    time.sleep(0.5)
                    st.rerun()


# --------------------------------------------------
# 朋友名單
# --------------------------------------------------

st.divider()
st.subheader("👥 朋友名單")

if not friends:
    if search_text:
        st.info("沒有找到符合搜尋條件的朋友。")
    else:
        st.info("目前尚未加入朋友。")
else:
    for friend in friends:
        with st.container(border=True):
            info_col, edit_col, delete_col = st.columns(
                [5, 1, 1]
            )

            with info_col:
                st.markdown(
                    f"### {friend['name']}"
                )

                st.write(
                    f"📍 {friend['city']}, "
                    f"{friend['country']}"
                )

                if friend.get("notes"):
                    st.caption(friend["notes"])

            with edit_col:
                if st.button(
                    "✏️ 編輯",
                    key=f"edit-{friend['id']}",
                    use_container_width=True,
                ):
                    st.session_state.editing_friend_id = (
                        friend["id"]
                    )
                    st.session_state.deleting_friend_id = None
                    st.rerun()

            with delete_col:
                if st.button(
                    "🗑️ 刪除",
                    key=f"delete-{friend['id']}",
                    use_container_width=True,
                ):
                    st.session_state.deleting_friend_id = (
                        friend["id"]
                    )
                    st.session_state.editing_friend_id = None
                    st.rerun()

            if (
                st.session_state.deleting_friend_id
                == friend["id"]
            ):
                st.warning(
                    f"確定要刪除 {friend['name']} 嗎？"
                )

                confirm_col, cancel_col = st.columns(2)

                with confirm_col:
                    if st.button(
                        "確認刪除",
                        key=f"confirm-delete-{friend['id']}",
                        type="primary",
                        use_container_width=True,
                    ):
                        delete_friend(friend["id"])
                        st.session_state.deleting_friend_id = None
                        st.success("朋友資料已刪除。")
                        time.sleep(0.5)
                        st.rerun()

                with cancel_col:
                    if st.button(
                        "取消",
                        key=f"cancel-delete-{friend['id']}",
                        use_container_width=True,
                    ):
                        st.session_state.deleting_friend_id = None
                        st.rerun()

Overwriting app.py
